# Chapter 09: How Transformers Work
This notebook accompanies Chapter 9 of *Deep Learning with PyTorch*. We implement character-level language modeling, autoregressive sequence data loaders, scaled dot-product causal attention, GPT-style decoder blocks, text generation, and Vision Transformer (ViT) patch extraction.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Set seeds for reproducibility and configure compute device
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")

## 2. Character Vocabulary & Tokenizer
We map characters to integer tokens using a bidirectional lookup table with a sentinel character `$` serving as start/end of sequence.

In [ ]:
special_char = '$'
alphabet = [chr(i) for i in range(ord('a'), ord('z') + 1)]
vocab = [special_char] + alphabet
vocab_size = len(vocab)

stoi = {char: idx for idx, char in enumerate(vocab)}
itos = {idx: char for idx, char in enumerate(vocab)}

print(f"Vocab Size: {vocab_size} | First 5: {vocab[:5]}")
sample_str = "$sada$"
encoded = [stoi[c] for c in sample_str]
decoded = ''.join([itos[i] for i in encoded])
print(f"Encoded '{sample_str}': {encoded}")
print(f"Decoded back: '{decoded}'")

## 3. Bigram Probability Matrix Baseline
A first-order Markov chain models transition probabilities: $P(x_t \mid x_{t-1}) = \frac{N(x_{t-1}, x_t)}{\sum_k N(x_{t-1}, x_k)}$.

In [ ]:
names_corpus = [
    "sada", "john", "emma", "olivia", "liam", "noah", "ava", "lucas",
    "sophia", "mason", "isabella", "ethan", "mia", "alexander", "charlotte"
]

bigram_counts = torch.zeros((vocab_size, vocab_size), dtype=torch.int32)
for name in names_corpus:
    seq = special_char + name + special_char
    for c1, c2 in zip(seq[:-1], seq[1:]):
        bigram_counts[stoi[c1], stoi[c2]] += 1

# Normalize with Laplace smoothing
bigram_probs = (bigram_counts.float() + 1.0)
bigram_probs /= bigram_probs.sum(dim=1, keepdim=True)

print("Top 3 characters likely to follow '$':")
top_indices = torch.topk(bigram_probs[stoi['$']], k=3).indices
for idx in top_indices:
    print(f"  '{itos[idx.item()]}': {bigram_probs[stoi['$'], idx]:.4f}")

## 4. Autoregressive Dataset with Sliding Prefixes
Each name is decomposed into expanding context prefixes paired with the immediate subsequent token target.

In [ ]:
class CharDataset(Dataset):
    def __init__(self, names, stoi, block_size=8):
        self.block_size = block_size
        self.inputs, self.targets = [], []
        for name in names:
            enc = [stoi[special_char]] + [stoi[c] for c in name] + [stoi[special_char]]
            for i in range(1, len(enc)):
                subseq = enc[:i]
                target = enc[i]
                if len(subseq) > block_size:
                    subseq = subseq[-block_size:]
                else:
                    subseq = [stoi[special_char]] * (block_size - len(subseq)) + subseq
                self.inputs.append(torch.tensor(subseq, dtype=torch.long))
                self.targets.append(torch.tensor(target, dtype=torch.long))
        self.inputs = torch.stack(self.inputs)
        self.targets = torch.stack(self.targets)

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

train_ds = CharDataset(names_corpus, stoi, block_size=8)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
bx, by = next(iter(train_loader))
print(f"Sample Batch X: {bx.shape} | Batch Y: {by.shape}")

## 5. Continuous Embedding Lookup
`nn.Embedding` provides an $O(1)$ memory lookup into a continuous trainable weight matrix $W_E \in \mathbb{R}^{|\mathcal{V}| \times d_{\text{model}}}$.

In [ ]:
d_model = 64
emb_layer = nn.Embedding(vocab_size, d_model)
embedded_batch = emb_layer(bx)
print(f"Embedded Tensor Shape: {embedded_batch.shape} -> (Batch, Block_Size, d_model)")

## 6. Scaled Dot-Product Causal Self-Attention
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right) V$$

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, block_size, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        self.c_attn = nn.Linear(d_model, 3 * d_model)
        self.c_proj = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        
        mask = torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
        self.register_buffer("causal_mask", mask, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(self.d_model, dim=2)
        q = q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.d_k))
        att = att.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))

attn_layer = CausalSelfAttention(d_model=64, n_heads=4, block_size=8)
attn_out = attn_layer(embedded_batch)
print(f"Attention Output Shape: {attn_out.shape}")

## 7. GPT Decoder Block & Complete Language Model
We assemble Pre-LayerNorm Transformer blocks with residual connections and a position-wise feed-forward network.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout)
        )
    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, block_size, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = FeedForward(d_model, dropout)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=3, block_size=8, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.blocks = nn.Sequential(*[TransformerBlock(d_model, n_heads, block_size, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.token_emb.weight = self.lm_head.weight  # Weight tying

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(0, T, device=idx.device)
        x = self.token_emb(idx) + self.pos_emb(pos)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            if targets.dim() == 1:
                loss = F.cross_entropy(logits[:, -1, :], targets)
            else:
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

model = GPTLanguageModel(vocab_size, d_model=64, n_heads=4, n_layers=3, block_size=8).to(device)
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"GPT Model Instantiated | Trainable Parameters: {params:,}")

## 8. Training Loop & Text Generation
We train on our name corpus and sample new names using autoregressive temperature decoding.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
model.train()

for epoch in range(1, 101):
    total_loss = 0.0
    for x_b, y_b in train_loader:
        x_b, y_b = x_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        logits, _ = model(x_b)
        loss = F.cross_entropy(logits[:, -1, :], y_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if epoch % 25 == 0 or epoch == 1:
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch:3d} | CrossEntropy Loss: {avg_loss:.4f}")

# Autoregressive text generation function
@torch.inference_mode()
def generate_name(model, prompt="$", max_len=12, temperature=0.8):
    model.eval()
    idx = torch.tensor([[stoi[c] for c in prompt]], dtype=torch.long, device=device)
    for _ in range(max_len):
        cond = idx[:, -model.block_size:]
        logits, _ = model(cond)
        logits = logits[:, -1, :] / max(temperature, 1e-5)
        probs = F.softmax(logits, dim=-1)
        next_tok = torch.multinomial(probs, 1)
        idx = torch.cat((idx, next_tok), dim=1)
        if next_tok.item() == stoi[special_char]:
            break
    return ''.join([itos[i.item()] for i in idx[0]])

print("\nSample Generated Names:")
for _ in range(5):
    gen = generate_name(model, prompt="$", temperature=0.7)
    print(f"  {gen}")

## 9. Vision Transformer (ViT) Patch Projection
A 2D image is partitioned into non-overlapping $P \times P$ patches using a 2D convolution with `stride=patch_size`.

In [ ]:
class ViTPatchEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.n_patches + 1, embed_dim))

    def forward(self, x):
        B = x.shape[0]
        x = self.proj(x).flatten(2).transpose(1, 2)  # (B, N, embed_dim)
        cls = self.cls_token.expand(B, -1, -1)        # (B, 1, embed_dim)
        x = torch.cat((cls, x), dim=1)               # (B, N+1, embed_dim)
        x = x + self.pos_embed
        return x

vit_embed = ViTPatchEmbedding(img_size=224, patch_size=16, in_channels=3, embed_dim=768)
dummy_image = torch.randn(2, 3, 224, 224)
vit_tokens = vit_embed(dummy_image)
print(f"ViT Input Shape:  {dummy_image.shape}")
print(f"ViT Token Output: {vit_tokens.shape} -> (Batch=2, N+1=197, embed_dim=768)")